# ARIS Monitor — Analisi Esplorativa (EDA) del Dataset
## `features_all.csv`: 89.682 vettori, 16 runnable, set 2024 – feb 2026

Questo notebook esplora il dataset prodotto dal pipeline_runner:
1. **Panoramica generale** — shape, distribuzione per runnable, copertura feature
2. **Segnale per runnable** — chi ha errori, chi è silenzioso
3. **Focus arcm_s** — crash noti, timeline, pattern temporali
4. **Correlazioni** — feature ridondanti vs. complementari
5. **Cross-runnable** — effetti a cascata tra servizi
6. **Conclusioni per il modello** — cosa sappiamo prima di addestrare

> **Nota**: il percorso del CSV va adattato al proprio ambiente.  
> Su WSL: `~/Azienda/Use_Case_2/features_all.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

# ── Configurazione ──
# Cambia questo percorso se necessario
CSV_PATH = "../features_all.csv"
# Alternativa WSL: CSV_PATH = "~/Azienda/Use_Case_2/features_all.csv"

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

df = pd.read_csv(CSV_PATH)
df['ts'] = pd.to_datetime(df['window_end'])
print(f"Dataset caricato: {df.shape[0]} righe × {df.shape[1]} colonne")
print(f"Runnable unici: {df['runnable'].nunique()}")
print(f"Time range: {df['ts'].min()} → {df['ts'].max()}")

---
## 1. Panoramica generale

### 1.1 Vettori per runnable
Ogni vettore rappresenta una finestra di 5 minuti per un runnable.
Il filtro "meaningful" nel pipeline_runner tiene solo i vettori con
almeno un segnale oppure un campione orario di baseline.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Vettori per runnable
rc = df['runnable'].value_counts().sort_values()
colors = ['#e74c3c' if r == 'arcm_s' else '#3498db' if r == 'agent' else '#95a5a6' 
          for r in rc.index]
rc.plot.barh(ax=axes[0], color=colors)
axes[0].set_title('Vettori per runnable', fontweight='bold')
axes[0].set_xlabel('Numero vettori')

# % vettori con errori per runnable
error_pct = []
for r in sorted(df['runnable'].unique()):
    sub = df[df['runnable'] == r]
    pct = (sub['error_count_5m'] > 0).sum() / len(sub) * 100
    error_pct.append((r, pct))
edf = pd.DataFrame(error_pct, columns=['runnable', 'pct']).sort_values('pct')
colors9 = ['#e74c3c' if p > 2 else '#f39c12' if p > 0.5 else '#2ecc71' for p in edf['pct']]
edf.plot.barh(x='runnable', y='pct', ax=axes[1], color=colors9, legend=False)
axes[1].set_title('% vettori con errori per runnable', fontweight='bold')
axes[1].set_xlabel('% con error_count_5m > 0')

plt.tight_layout()
plt.show()

### 1.2 Distribuzione stati
Solo il runnable `agent` ha stati diversi da UNKNOWN — perché i record
di `runnable.history.log` vengono attribuiti all'agent. Gli altri
runnable hanno `current_state = UNKNOWN` perché i loro stati sono
nel log dell'agent, non nei propri log.

In [ ]:
agent = df[df['runnable'] == 'agent']
state_counts = agent['current_state'].value_counts()

state_colors = {
    'STARTED': '#2ecc71', 'FAILED': '#e74c3c', 'STARTING': '#f39c12',
    'STOPPED': '#e67e22', 'STOPPING': '#9b59b6', 'DEACTIVATED': '#34495e', 
    'UNKNOWN': '#bdc3c7'
}

fig, ax = plt.subplots(figsize=(8, 8))
sc_colors = [state_colors.get(s, '#bdc3c7') for s in state_counts.index]
state_counts.plot.pie(ax=ax, colors=sc_colors, autopct='%1.1f%%', startangle=90)
ax.set_title('agent — Distribuzione stati', fontweight='bold', fontsize=14)
ax.set_ylabel('')
plt.show()

print("Conteggi:")
for state, count in state_counts.items():
    print(f"  {state:15s}: {count:>6d} ({count/len(agent)*100:.1f}%)")

### 1.3 Copertura feature infrastrutturali
Le feature derivate dai CSV monitordata (heap, cache, login, HTTP pool)
sono **tutte a -1** — i CSV non sono ancora integrati nel pipeline_runner.
Questa è la lacuna #7 del handoff, e riempirla darà al modello feature
reali sull'infrastruttura JVM.

In [ ]:
infra_features = ['heap_committed_mb', 'heap_used_pct', 'cache_avg_get_time_ms',
                  'login_count_5m', 'login_mean_duration_ms', 'login_p95_duration_ms',
                  'login_rate_per_sec', 'http_pool_available', 'http_pool_leased', 
                  'http_pool_pending']

print("Feature infrastrutturali — copertura (valori ≠ -1):")
print("-" * 55)
for f in infra_features:
    has_data = (df[f] != -1).sum()
    pct = has_data / len(df) * 100
    status = "✓ DATI" if has_data > 0 else "✗ VUOTA"
    print(f"  {f:35s}: {has_data:>6d} ({pct:5.2f}%) {status}")

---
## 2. Segnale per runnable — Chi ha errori, chi è silenzioso

Analizziamo quali feature log hanno valori non-zero per ciascun runnable.
Questo ci dice dove il modello troverà segnale utile.

In [ ]:
log_feats = ['error_count_5m', 'error_count_1h', 'fatal_count_1h', 'warn_count_1h',
             'has_stack_trace_5m', 'state_transitions_1h', 'restarts_24h',
             'pool_exhaustion_count_5m', 'bus_error_count_5m', 'zkc_error_count_5m']

# Heatmap: % non-zero per runnable × feature
heatmap_data = []
runnables_sorted = sorted(df['runnable'].unique())
for r in runnables_sorted:
    sub = df[df['runnable'] == r]
    row = [(sub[f] != 0).sum() / len(sub) * 100 for f in log_feats]
    heatmap_data.append(row)

hm = pd.DataFrame(heatmap_data, index=runnables_sorted, 
                   columns=[f.replace('_count', '').replace('_5m', '').replace('_1h', '') 
                           for f in log_feats])

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(hm, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax, 
            linewidths=0.5, cbar_kws={'label': '% vettori non-zero'})
ax.set_title('% vettori con segnale per runnable × feature', fontweight='bold', fontsize=13)
ax.set_ylabel('Runnable')
plt.tight_layout()
plt.show()

**Osservazioni:**
- **arcm_s** è il più ricco: errori, fatal, pool exhaustion, BUS errors, stack trace, ZKC
- **abs_s** ha molti warning (44%) e errori (22.6%)
- **agent** domina su `restarts_24h` (55%) e `state_transitions` (4%) — è il "sentinella"
- **zoo_s, elastic_s, cloudsearch_s** hanno solo restart — sono i servizi fondamentali e stabili
- `restarts_24h` è non-zero in quasi tutti i runnable (24-55%) — è una feature di "memoria lunga" 

---
## 3. Focus arcm_s — Crash noti e timeline

arcm_s è il runnable con il segnale più ricco e i crash meglio documentati.
Abbiamo 5 crash noti che servono come ground truth per la validazione.

In [ ]:
arcm = df[df['runnable'] == 'arcm_s'].copy()
arcm_ts = arcm.set_index('ts')

fig, axes = plt.subplots(3, 1, figsize=(18, 14), sharex=True)

# Timeline errori
axes[0].fill_between(arcm_ts.index, arcm_ts['error_count_5m'], 
                     alpha=0.8, color='#e74c3c', label='error_count_5m')
axes[0].set_ylabel('Errori / 5min')
axes[0].set_title('arcm_s — Timeline completa con crash noti', fontweight='bold', fontsize=13)
axes[0].legend(loc='upper right')

# Pool exhaustion + BUS
axes[1].fill_between(arcm_ts.index, arcm_ts['pool_exhaustion_count_5m'], 
                     alpha=0.8, color='#2980b9', label='pool_exhaustion_5m')
axes[1].fill_between(arcm_ts.index, arcm_ts['bus_error_count_5m'], 
                     alpha=0.5, color='#27ae60', label='bus_error_5m')
axes[1].set_ylabel('Conteggio / 5min')
axes[1].legend(loc='upper right')

# Fatal + warn
axes[2].fill_between(arcm_ts.index, arcm_ts['fatal_count_1h'], 
                     alpha=0.8, color='#8e44ad', label='fatal_count_1h')
axes[2].fill_between(arcm_ts.index, arcm_ts['warn_count_1h'], 
                     alpha=0.5, color='#f39c12', label='warn_count_1h')
axes[2].set_ylabel('Conteggio / 1h')
axes[2].legend(loc='upper right')

# Crash markers su tutti i pannelli
crash_dates = [
    ('2025-11-10', 'C1: Migration\nrestart loop'),
    ('2025-12-03', 'C2: Pool\nexhaustion'),
    ('2025-12-18', 'C3: Pool\n6 giorni'),
    ('2026-01-15', 'C4: Pool\nrecurrence'),
    ('2026-02-20', 'C5: NPE +\nBulkExport'),
]
for ax in axes:
    for date, label in crash_dates:
        ax.axvline(pd.Timestamp(date), color='black', linestyle='--', alpha=0.5, lw=0.8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

# Labels solo sul primo pannello
for date, label in crash_dates:
    axes[0].text(pd.Timestamp(date), axes[0].get_ylim()[1]*0.75, label, 
                rotation=0, fontsize=7, ha='left', va='top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.show()

### 3.1 Film dei crash — Zoom sugli eventi

In [ ]:
crash_windows = [
    ("Crash 1 — 10 Nov 2025: Migration failure restart loop",
     "2025-11-10 08:00", "2025-11-10 14:00",
     ['error_count_5m', 'fatal_count_1h', 'has_stack_trace_5m', 'restarts_24h']),
    ("Crash 2 — 3-4 Dec 2025: Pool exhaustion early warnings",
     "2025-12-03 00:00", "2025-12-04 06:00",
     ['error_count_5m', 'pool_exhaustion_count_5m', 'bus_error_count_5m', 'has_stack_trace_5m']),
    ("Crash 3 — 18 Dec 2025: Pool exhaustion onset",
     "2025-12-17 20:00", "2025-12-18 08:00",
     ['error_count_5m', 'pool_exhaustion_count_5m', 'bus_error_count_5m', 'error_count_1h']),
    ("Crash 5 — 20 Feb 2026: NPE + BulkExport",
     "2026-02-20 12:00", "2026-02-20 18:00",
     ['error_count_5m', 'has_stack_trace_5m', 'error_count_1h']),
]

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.flatten()
colors_cycle = ['#e74c3c', '#2980b9', '#27ae60', '#8e44ad']

for i, (title, start, end, feats) in enumerate(crash_windows):
    ax = axes[i]
    window = arcm[(arcm['ts'] >= start) & (arcm['ts'] <= end)].set_index('ts')
    
    for j, f in enumerate(feats):
        ax.plot(window.index, window[f], '-o', markersize=3, color=colors_cycle[j % 4],
                label=f.replace('_count', '').replace('_5m', '5m').replace('_1h', '1h'),
                linewidth=1.5)
    
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.legend(fontsize=7, loc='upper right')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %H:%M'))
    ax.tick_params(axis='x', rotation=30, labelsize=8)

plt.tight_layout()
plt.show()

### 3.2 Tabella crash — Valori di picco

| Crash | Data | Tipo | error_5m max | fatal_1h max | pool_5m max | bus_5m max | stack_5m max |
|-------|------|------|:---:|:---:|:---:|:---:|:---:|
| C1 | 10 Nov 2025 | Migration failure | 47 | 45 | 0 | 0 | 11 |
| C2 | 3-4 Dec 2025 | Pool exhaustion | 57 | 0 | 17 | 32 | 22 |
| C3 | 18-23 Dec 2025 | Pool sustained | 83 | 0 | 24 | 41 | 59 |
| C4 | 15-16 Gen 2026 | Pool recurrence | da verificare | | | | |
| C5 | 20 Feb 2026 | NPE + BulkExport | 14 | 0 | 0 | 0 | 10 |

**Nota**: C1 ha fatal ma zero pool/bus — è un pattern diverso (migration).
C2-C3-C4 hanno pool+bus ma zero fatal — è il pattern pool exhaustion.
C5 è il più piccolo: 14 errori in 5 min, solo stack trace.

---
## 4. Correlazioni tra feature

Analizziamo le correlazioni per capire quali feature sono ridondanti
e quali portano informazione complementare.

In [ ]:
num_feats = ['error_count_5m', 'error_count_1h', 'fatal_count_1h', 'warn_count_1h',
             'distinct_errors_5m', 'error_rate_delta', 'has_stack_trace_5m',
             'state_transitions_1h', 'restarts_24h', 'pool_exhaustion_count_5m',
             'bus_error_count_5m', 'zkc_error_count_5m']

corr = arcm[num_feats].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.5,
            xticklabels=[f.replace('_count', '').replace('_5m', '') for f in num_feats],
            yticklabels=[f.replace('_count', '').replace('_5m', '') for f in num_feats])
ax.set_title('arcm_s — Matrice di correlazione feature log', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.show()

# Coppie altamente correlate
print("\nCoppie con |r| > 0.90 (quasi ridondanti):")
for i in range(len(num_feats)):
    for j in range(i+1, len(num_feats)):
        r = corr.iloc[i, j]
        if abs(r) > 0.90:
            print(f"  {num_feats[i]:35s} ↔ {num_feats[j]:35s}: r={r:+.3f}")

**Interpretazione:**

**Cluster "pool exhaustion"** (r > 0.97 tra loro):
- `error_count_5m`, `has_stack_trace_5m`, `pool_exhaustion_5m`, `bus_error_5m`
- Sono aspetti diversi dello stesso evento: il pool si satura → BUS error → stack trace → conteggio errore
- Per Isolation Forest **non è un problema** (IF non è sensibile alla collinearità)
- Per un Autoencoder potrebbe creare bias → valutare PCA o feature selection

**Feature complementari** (correlazione bassa con le altre):
- `fatal_count_1h` — segnala solo il Crash 1 (migration), indipendente dal cluster pool
- `restarts_24h` — memoria lunga, non correlata ai burst brevi
- `state_transitions_1h` — presente solo nell'agent
- `error_rate_delta` — cattura l'onset (inizio) di un burst, non la durata

---
## 5. Pattern temporali

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Errori per ora del giorno
arcm_h = arcm.copy()
arcm_h['hour'] = arcm_h['ts'].dt.hour
hourly_err = arcm_h.groupby('hour')['error_count_5m'].sum()
full_hours = pd.Series(0.0, index=range(24))
full_hours.update(hourly_err)
bar_colors = ['#e74c3c' if v > 200 else '#f39c12' if v > 50 else '#3498db' for v in full_hours]
full_hours.plot.bar(ax=axes[0], color=bar_colors)
axes[0].set_title('arcm_s — Errori per ora UTC', fontweight='bold')
axes[0].set_xlabel('Ora UTC')
axes[0].set_ylabel('Somma error_count_5m')

# Errori per giorno della settimana
arcm_h['dow'] = arcm_h['ts'].dt.dayofweek
dow_err = arcm_h.groupby('dow')['error_count_5m'].sum()
dow_names = ['Lun', 'Mar', 'Mer', 'Gio', 'Ven', 'Sab', 'Dom']
dow_err.index = [dow_names[i] for i in dow_err.index]
dow_err.plot.bar(ax=axes[1], color='#3498db')
axes[1].set_title('arcm_s — Errori per giorno della settimana', fontweight='bold')
axes[1].set_ylabel('Somma error_count_5m')

plt.tight_layout()
plt.show()

print("Nota: le ore sono in UTC. L'Italia è UTC+1 (CET) o UTC+2 (CEST).")
print("Gli errori si concentrano nelle ore 00-03 UTC = 01-04 CET → batch job notturni.")

---
## 6. Analisi cross-runnable — Effetti a cascata

Quando arcm_s ha problemi, anche altri runnable li hanno?

In [ ]:
fig, ax = plt.subplots(figsize=(18, 6))

top_runnables = ['arcm_s', 'abs_s', 'agent', 'copernicus_s', 'ces_s', 
                 'dashboarding_s', 'umcadmin_s', 'ecp_s']
palette = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12', '#1abc9c', '#e67e22', '#34495e']

for i, r in enumerate(top_runnables):
    sub = df[(df['runnable'] == r) & (df['error_count_5m'] > 0)].copy()
    if len(sub) > 0:
        sizes = sub['error_count_5m'].clip(upper=60) * 2
        ax.scatter(sub['ts'], [i]*len(sub), s=sizes, alpha=0.6, 
                  color=palette[i], label=f"{r} ({len(sub)})")

# Crash lines
for date, _ in [('2025-11-10',''), ('2025-12-03',''), ('2025-12-18',''), ('2026-02-20','')]:
    ax.axvline(pd.Timestamp(date), color='black', linestyle='--', alpha=0.3, lw=0.8)

ax.set_yticks(range(len(top_runnables)))
ax.set_yticklabels(top_runnables, fontsize=9)
ax.set_title('Errori cross-runnable nel tempo (dimensione ∝ intensità)', fontweight='bold', fontsize=13)
ax.legend(loc='upper right', fontsize=8, ncol=2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.tight_layout()
plt.show()

In [ ]:
# Giorni in cui arcm_s ha errori: chi altro ne ha?
arcm_err_dates = set(arcm[arcm['error_count_5m'] > 5]['ts'].dt.date)

print("Giorni con arcm_s error_count_5m > 5 — altri runnable coinvolti:")
print("-" * 70)
for date in sorted(arcm_err_dates):
    day_data = df[df['ts'].dt.date == date]
    active = day_data[day_data['error_count_5m'] > 0].groupby('runnable')['error_count_5m'].sum()
    active = active[active > 0].sort_values(ascending=False)
    others = [f"{r}({int(v)})" for r, v in active.items() if r != 'arcm_s']
    arcm_val = active.get('arcm_s', 0)
    print(f"  {date}: arcm_s({int(arcm_val)}) + {', '.join(others) if others else 'nessun altro'}")

**Osservazioni cross-runnable:**
- Il **5 novembre 2025** è un evento globale: errori in 10 runnable su 16 → probabile restart/deploy del server
- Il **3 dicembre 2025** colpisce arcm_s (pool) ma anche abs_s (50.573 warning!) → effetto a cascata
- **copernicus_s** ha burst isolati (19 dic, 30 gen) non correlati ad arcm_s
- **ces_s** ha un burst massiccio il 12 feb 2026, non correlato ad arcm_s

→ Il modello dovrebbe essere addestrato **per-runnable** ma con la possibilità di aggiungere feature cross-runnable in futuro.

---
## 7. Conclusioni per il modello

### Cosa sappiamo:
1. **Il dataset è estremamente sbilanciato** — <5% dei vettori hanno errori. Perfetto per Isolation Forest (unsupervised anomaly detection).

2. **arcm_s è il runnable di riferimento** per la validazione — ha 5 crash documentati con pattern diversi (migration vs pool exhaustion vs NPE).

3. **Due macro-pattern di crash:**
   - **Migration failure** (C1): fatal+warn esplosivi, zero pool/bus → si manifesta come spike di fatal_count_1h
   - **Pool exhaustion** (C2, C3, C4): pool+bus+stack correlati al 98%, zero fatal → si manifesta come cluster pool_exhaustion + bus_error

4. **Feature utili per il modello** (12 log-derived):
   - `error_count_5m`, `error_count_1h` — segnale primario
   - `fatal_count_1h` — discrimina il pattern migration
   - `pool_exhaustion_count_5m`, `bus_error_count_5m` — discriminano il pattern pool
   - `error_rate_delta` — cattura l'onset
   - `has_stack_trace_5m` — gravità
   - `restarts_24h` — memoria lunga
   - `distinct_errors_5m` — diversità degli errori
   - `warn_count_1h`, `state_transitions_1h`, `zkc_error_count_5m` — segnali secondari

5. **Feature infrastrutturali VUOTE** — le 10 feature da CSV monitordata sono tutte -1. Integrarle (lacuna #7) aggiungerà heap, cache, login, HTTP pool → potenziale miglioramento significativo per abs_s.

6. **Pattern temporale** — errori concentrati 00-03 UTC (batch notturni CET).

### Prossimi passi:
- **Addestramento Isolation Forest** su tutti i 16 runnable
- **Validazione retrospettiva** sui 5 crash noti
- **Confronto con Autoencoder** 
- **Integrazione CSV monitordata** per le feature infrastrutturali